In [1]:
from dotenv import load_dotenv

_ = load_dotenv()

In [2]:
from langgraph.graph import StateGraph, END
from typing import TypedDict, Annotated
import operator
from langchain_core.messages import AnyMessage, SystemMessage, HumanMessage, ToolMessage
from langchain_core.tools import tool
from langchain_ollama import ChatOllama
from langchain_tavily import TavilySearch

In [3]:
_tavily = TavilySearch(max_results=2)

@tool
def tavily_search(query: str) -> str:
    """Search the web for current information about a topic."""
    return str(_tavily.invoke({"query": query}))

tool_instance = tavily_search

In [4]:
class AgentState(TypedDict):
    messages: Annotated[list[AnyMessage], operator.add]

In [5]:
from langgraph.checkpoint.memory import MemorySaver

memory = MemorySaver()

In [6]:
class Agent:
    def __init__(self, model, tools, checkpointer, system=""):
        self.system = system
        graph = StateGraph(AgentState)
        graph.add_node("llm", self.call_ollama)
        graph.add_node("action", self.take_action)
        graph.add_conditional_edges("llm", self.exists_action, {True: "action", False: END})
        graph.add_edge("action", "llm")
        graph.set_entry_point("llm")
        self.graph = graph.compile(checkpointer=checkpointer)
        self.tools = {t.name: t for t in tools}
        self.model = model.bind_tools(tools)

    def call_ollama(self, state: AgentState):
        messages = state['messages']
        if self.system:
            messages = [SystemMessage(content=self.system)] + messages
        message = self.model.invoke(messages)
        return {'messages': [message]}

    def exists_action(self, state: AgentState):
        result = state['messages'][-1]
        return len(result.tool_calls) > 0

    def take_action(self, state: AgentState):
        tool_calls = state['messages'][-1].tool_calls
        results = []
        for t in tool_calls:
            print(f"Calling: {t}")
            args = {"query": t['args']['query']}
            result = self.tools[t['name']].invoke(args)
            results.append(ToolMessage(tool_call_id=t['id'], name=t['name'], content=str(result)))
        print("Back to the model!")
        return {'messages': results}

In [7]:
prompt = """You are a smart research assistant. Use the search engine to look up information. \
You are allowed to make multiple calls (either together or in sequence). \
Only look up information when you are sure of what you want. \
If you need to look up some information before asking a follow up question, you are allowed to do that!
"""
model = ChatOllama(model="llama3.1:8b")
abot = Agent(model, [tool_instance], system=prompt, checkpointer=memory)

In [8]:
messages = [HumanMessage(content="what is the weather in seoul?")]

In [9]:
thread = {"configurable": {"thread_id": "1"}}

In [10]:
for event in abot.graph.stream({"messages": messages}, thread):
    for v in event.values():
        print(v['messages'])

[AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'llama3.1:8b', 'created_at': '2026-06-09T02:08:46.289602Z', 'done': True, 'done_reason': 'stop', 'total_duration': 936690625, 'load_duration': 94047417, 'prompt_eval_count': 223, 'prompt_eval_duration': 400152917, 'eval_count': 21, 'eval_duration': 414457663, 'logprobs': None, 'model_name': 'llama3.1:8b', 'model_provider': 'ollama'}, id='lc_run--019eaa23-ad23-7cf1-ac46-bfab9964cfb6-0', tool_calls=[{'name': 'tavily_search', 'args': {'query': 'seoul weather'}, 'id': 'deaee3e5-dd87-4719-95da-b4ff29ec6e6d', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 223, 'output_tokens': 21, 'total_tokens': 244})]
Calling: {'name': 'tavily_search', 'args': {'query': 'seoul weather'}, 'id': 'deaee3e5-dd87-4719-95da-b4ff29ec6e6d', 'type': 'tool_call'}
Back to the model!
[ToolMessage(content='{\'query\': \'seoul weather\', \'follow_up_questions\': None, \'answer\': None, \'images\': [], \'results\': [{\

In [11]:
messages = [HumanMessage(content="what is the weather in jeju?")]
thread = {"configurable": {"thread_id": "1"}}
for event in abot.graph.stream({"messages": messages}, thread):
    for v in event.values():
        print(v)

{'messages': [AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'llama3.1:8b', 'created_at': '2026-06-09T02:08:59.868765Z', 'done': True, 'done_reason': 'stop', 'total_duration': 947815083, 'load_duration': 106723583, 'prompt_eval_count': 3201, 'prompt_eval_duration': 375395833, 'eval_count': 21, 'eval_duration': 444229252, 'logprobs': None, 'model_name': 'llama3.1:8b', 'model_provider': 'ollama'}, id='lc_run--019eaa23-e224-7c21-bd0f-8a0aa0cc58be-0', tool_calls=[{'name': 'tavily_search', 'args': {'query': 'jeju weather'}, 'id': 'e8bde378-2f79-458f-a8cc-cdc38dcc1154', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 3201, 'output_tokens': 21, 'total_tokens': 3222})]}
Calling: {'name': 'tavily_search', 'args': {'query': 'jeju weather'}, 'id': 'e8bde378-2f79-458f-a8cc-cdc38dcc1154', 'type': 'tool_call'}
Back to the model!
{'messages': [ToolMessage(content='{\'query\': \'jeju weather\', \'follow_up_questions\': None, \'answer\': None, \'im

In [12]:
messages = [HumanMessage(content="Which one is warmer?")]
thread = {"configurable": {"thread_id": "1"}}
for event in abot.graph.stream({"messages": messages}, thread):
    for v in event.values():
        print(v)

{'messages': [AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'llama3.1:8b', 'created_at': '2026-06-09T02:09:17.447785Z', 'done': True, 'done_reason': 'stop', 'total_duration': 1082221708, 'load_duration': 91127917, 'prompt_eval_count': 3919, 'prompt_eval_duration': 434267833, 'eval_count': 25, 'eval_duration': 541488460, 'logprobs': None, 'model_name': 'llama3.1:8b', 'model_provider': 'ollama'}, id='lc_run--019eaa24-264c-70a3-8da3-7d02504b3d30-0', tool_calls=[{'name': 'tavily_search', 'args': {'query': 'jeju vs seoul temperature comparison'}, 'id': '2595cc81-3ad8-4312-b80d-30d28bf0d69c', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 3919, 'output_tokens': 25, 'total_tokens': 3944})]}
Calling: {'name': 'tavily_search', 'args': {'query': 'jeju vs seoul temperature comparison'}, 'id': '2595cc81-3ad8-4312-b80d-30d28bf0d69c', 'type': 'tool_call'}
Back to the model!
{'messages': [ToolMessage(content='{\'query\': \'jeju vs seoul tempera

In [13]:
messages = [HumanMessage(content="Which one is warmer?")]
thread = {"configurable": {"thread_id": "2"}}
for event in abot.graph.stream({"messages": messages}, thread):
    for v in event.values():
        print(v)

{'messages': [AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'llama3.1:8b', 'created_at': '2026-06-09T02:09:30.775654Z', 'done': True, 'done_reason': 'stop', 'total_duration': 932824959, 'load_duration': 88560750, 'prompt_eval_count': 220, 'prompt_eval_duration': 356836209, 'eval_count': 24, 'eval_duration': 475622374, 'logprobs': None, 'model_name': 'llama3.1:8b', 'model_provider': 'ollama'}, id='lc_run--019eaa24-5aef-71e2-9fde-a527c23ba72c-0', tool_calls=[{'name': 'tavily_search', 'args': {'query': 'what is warmer summer or winter'}, 'id': 'cbc9d6d4-07ca-410a-bea0-3802b8a57fc2', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 220, 'output_tokens': 24, 'total_tokens': 244})]}
Calling: {'name': 'tavily_search', 'args': {'query': 'what is warmer summer or winter'}, 'id': 'cbc9d6d4-07ca-410a-bea0-3802b8a57fc2', 'type': 'tool_call'}
Back to the model!
{'messages': [ToolMessage(content="{'query': 'what is warmer summer or winter', 'fol

Streaming tokens

In [14]:
memory = MemorySaver()
abot = Agent(model, [tool_instance], system=prompt, checkpointer=memory)

In [15]:
messages = [HumanMessage(content="what is the weather in seoul?")]
thread = {"configurable": {"thread_id": "4"}}
async for event in abot.graph.astream_events({"messages": messages}, thread, version="v1"):
    kind = event["event"]
    if kind == "on_chat_model_stream":
        content = event["data"]["chunk"].content
        if content:
            print(content, end="|")

/Users/baejinho/Documents/GitHub/ai-agent-cloud/14주차/.venv/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3746: LangChainDeprecationWarning: astream_events version='v1' is deprecated. Use version='v2' or astream instead.
  await eval(code_obj, self.user_global_ns, self.user_ns)


Calling: {'name': 'tavily_search', 'args': {'query': 'weather in Seoul'}, 'id': '58d32366-fe8e-47ac-a551-384bf89c6077', 'type': 'tool_call'}
Back to the model!
The| current| weather| in| Seoul| is| sunny| with| a| temperature| of| |24|°C| (|75|.|2|°F|)| and| a| humidity| level| of| |47|%.| There| is| no| precipitation| expected|,| and| the| wind| speed| is| approximately| |5|.|1| mph| (|8|.|3| k|ph|).|